# Graph-based multi-hop retrieval — full pipeline

Thin launcher: all the logic lives in the repo (editable in an IDE, reviewable in PRs);
this notebook only clones it and runs stages `01` → `04` in order.

**Before running:**
* Colab — *Runtime → Change runtime type → T4 GPU*.
* Kaggle — *Settings → Accelerator → GPU*, *Internet → On*, then *Save & Run All (Commit)* for free background execution.

This is the real thing: the full 609-document MultiHop-RAG corpus with the models in
`config/base.yaml` (`bge-large-en-v1.5`, `en_core_web_lg`). Expect ~20–30 minutes on a T4,
most of it in stage 2 embedding edge texts. For a fast structural check instead, use
`colab_run_tests.ipynb`.

Every stage runs through `scripts/experiment.py`, which is what gives the run a name. This
notebook uses the id `default`, so everything lands in `results/default/` — metrics, the exact
config that produced them, and a log per stage. **To compare several configs — a
`mutual_knn_k` sweep, say — use `colab_run_experiments.ipynb` instead**; it declares parameter
lists and gives each arm its own `results/<id>/` folder.

**On Colab the clone disappears when the runtime ends** — run the last cell to download what
you care about.

## 1. Clone the repo

In [ ]:
REF = "main"  # branch for iteration, or a commit SHA to pin a run exactly
REPO_URL = "https://github.com/hadasy-tau/graphs_project.git"

import os

# Kaggle keeps writable state in /kaggle/working, Colab in /content, anywhere else: here.
BASE = next((d for d in ("/kaggle/working", "/content") if os.path.isdir(d)), os.getcwd())
REPO = os.path.join(BASE, "graphs_project")

if os.path.isdir(REPO):
    !cd {REPO} && git fetch --all --quiet && git checkout {REF} && git pull --ff-only || true
else:
    !git clone {REPO_URL} {REPO} && cd {REPO} && git checkout {REF}

os.chdir(REPO)  # every later cell, shell command included, runs from the repo root
!git log --oneline -1

## 2. Install dependencies

Two to three minutes. `pcst-fast` compiles from C++ source here — fine on Linux, and stage 3
cannot run without it.

In [ ]:
# requirements.txt pins the en_core_web_lg 3.7.1 wheel. That pin drags spaCy back to 3.7.x
# and numpy below 2.0 with it, a downgrade the already-running kernel only picks up after a
# restart. So install everything else from the file and let spaCy fetch the model build that
# matches whatever version it resolved - same NER model, no downgrade, no restart.
lines = [l for l in open("requirements.txt").read().splitlines() if "en-core-web" not in l]
with open("/tmp/requirements-colab.txt", "w") as f:
    f.write("\n".join(lines) + "\n")

!pip install -q -r /tmp/requirements-colab.txt
!python -m spacy download en_core_web_lg

In [ ]:
# Sanity check: fail here rather than 10 minutes into a stage.
import spacy
import torch

print("torch     :", torch.__version__)
print("GPU       :", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "NONE - Colab: Runtime > Change runtime type | Kaggle: Settings > Accelerator")

spacy.load("en_core_web_lg")
print("spaCy NER : en_core_web_lg OK")

try:
    import pcst_fast  # noqa: F401
    print("pcst_fast : OK (the real Goemans-Williamson solver)")
except ImportError:
    print("pcst_fast : MISSING - stage 3 will fail. Re-run `pip install pcst-fast` and read its error.")

In [ ]:
# What this run will use. Edit config/base.yaml before stage 1 to change it - or, to try
# several values without editing anything, use colab_run_experiments.ipynb.
print(open("config/base.yaml").read())

## 3. Stage 1 — data, embeddings, NER

Downloads MultiHop-RAG, cleans the bodies, embeds 609 documents, their metadata records and
the queries, and runs spaCy NER. First run also pulls the ~1.3 GB `bge-large-en-v1.5`
checkpoint. The results are cached in `data/processed/p-<fingerprint of the embedding, NER and
metadata-field config>/`, so re-running this stage afterwards is seconds.

In [ ]:
!python -u scripts/experiment.py --id default --stages 01

## 4. Stage 2 — build the four graphs

Entity, metadata, semantic, and their union. The slow part is embedding one text per edge for
PCST's edge scores, not deciding the edges themselves. The graphs are cached under
`data/graphs/g-<fingerprint of the graph config>/`, so a later run with the same graph settings
reuses them instead of paying that cost again.

In [ ]:
!python -u scripts/experiment.py --id default --stages 02

## 5. Stage 3 — retrieval

All ten conditions in one process: each graph with and without PCST, plus the dense baseline
with K calibrated to the average entity-PCST output size. The similarity matrix and each
graph's `seed_k` are computed once and shared across conditions. Writes
`results/default/retrieval/*.jsonl`, skipping any condition already written — so re-running
after a disconnect picks up where it stopped.

In [ ]:
!python -u scripts/experiment.py --id default --stages 03

## 6. Stage 4 — metrics

In [ ]:
!python -u scripts/experiment.py --id default --stages 04

## 7. Look at the results

In [ ]:
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

for csv in sorted(Path("results/default/metrics").glob("*.csv")):
    print()
    print(f"=== {csv} ===")
    display(pd.read_csv(csv))

## 8. Download the results

Bundles `results/default/` — retrieval JSONL, metrics, the log of every stage, and
`config.yaml`, the exact config this run consumed. The heavy intermediates under `data/` are
left behind on purpose; they are hundreds of MB and stage 1 regenerates them from cache.

In [ ]:
import shutil

BUNDLE = os.path.join(BASE, "graphs_project_results")
shutil.rmtree(BUNDLE, ignore_errors=True)
shutil.copytree("results/default", os.path.join(BUNDLE, "default"))

zip_path = shutil.make_archive(BUNDLE, "zip", BUNDLE)
print(f"{zip_path}  ({os.path.getsize(zip_path) / 1e6:.1f} MB)")

try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("Kaggle: download it from the Output tab.")